# UTTA-Med — Ablation Study A: MC-Dropout Pass Sensitivity

**Research Question**: How sensitive is UTTA-Med's predictive-variance gate to the number of MC-Dropout stochastic forward passes $N_{MC} \in \{5, 10, 20, 30, 50\}$?

- **Backbone**: ResNet-18
- **Checkpoint**: `camelyon17_resnet18_source_s42_BEST.pt` (Seed 42, frozen)
- **OOD Validation**: Center 1 (tune $\tau$ at 70th percentile of predictive variance $U$)
- **Target Test**: Center 2 (unlabeled adaptation stream, evaluated once with frozen $\tau$)
- **TTA Protocol**: `bn_freeze_stats`, Adam LR $10^{-5}$, entropy minimization on BN affine parameters

In [ ]:
!pip -q install datasets pyarrow scikit-learn tqdm pillow matplotlib pyyaml pandas scipy

In [ ]:
import glob, json, os, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from tqdm.auto import tqdm

WORK = Path('/kaggle/working' if os.path.exists('/kaggle') else './results/ablations/mc_passes')
WORK.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
EVAL_TF = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

PASS_GRID = [5, 10, 20, 30, 50]
TTA_LR = 1e-5
SEED = 42

In [ ]:
def find_ckpt():
    hits = glob.glob('/kaggle/input/**/*s42*BEST*.pt', recursive=True)
    hits += glob.glob('/kaggle/input/**/*source_s42*.pt', recursive=True)
    hits += glob.glob('checkpoints/*s42*BEST*.pt')
    if not hits: raise FileNotFoundError('Add dataset containing camelyon17_resnet18_source_s42_BEST.pt')
    print('CKPT:', hits[0])
    return hits[0]

CKPT = find_ckpt()

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        bb = models.resnet18(weights=None)
        d = bb.fc.in_features
        bb.fc = nn.Identity()
        self.backbone = bb
        self.classifier = nn.Sequential(nn.Linear(d, d), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(d, 1))
    def forward(self, x):
        return self.classifier(self.backbone(x))

def load_fresh():
    m = Net().to(device)
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    sd = ck['model'] if isinstance(ck, dict) and 'model' in ck else ck
    m.load_state_dict(sd, strict=True)
    return m

In [ ]:
# Load datasets: Center 1 (OOD val) and Center 2 (Target test)
print('Loading Camelyon17 from Hugging Face / Kaggle...')
ds_val = load_dataset('flwrlabs/camelyon17', split='val')
ds_test = load_dataset('flwrlabs/camelyon17', split='test')

class HFDataset(Dataset):
    def __init__(self, ds):
        self.ds = ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item['image'].convert('RGB') if isinstance(item['image'], Image.Image) else item['image']
        return EVAL_TF(img), torch.tensor(item['label'], dtype=torch.float32), item.get('slide', 0)

val_loader = DataLoader(HFDataset(ds_val), batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(HFDataset(ds_test), batch_size=128, shuffle=False, num_workers=2)
print('Val patches:', len(ds_val), 'Test patches:', len(ds_test))

In [ ]:
def bn_affine(m):
    for p in m.parameters(): p.requires_grad = False
    ps = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d) and mod.affine:
            mod.weight.requires_grad = True; mod.bias.requires_grad = True
            ps.extend([mod.weight, mod.bias])
    return ps

def eval_mode(m):
    m.eval()
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d): mod.eval()

def mc_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d): mod.eval()
        elif isinstance(mod, nn.Dropout): mod.train()

def tta_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.Dropout): mod.eval()
        elif isinstance(mod, nn.BatchNorm2d): mod.eval()

def entropy(z):
    p = torch.sigmoid(z).clamp(1e-6, 1.0 - 1e-6)
    return -(p * torch.log(p) + (1.0 - p) * torch.log(1.0 - p)).view(-1)

@torch.no_grad()
def mc_predict(m, x, n):
    mc_mode(m)
    ps = [torch.sigmoid(m(x)).view(-1) for _ in range(n)]
    P = torch.stack(ps, 0)
    return P.mean(0), P.var(0, unbiased=False)

In [ ]:
# Execute Ablation over N_MC in {5, 10, 20, 30, 50}
results = []
u_test_dict = {}
mask_test_dict = {}

for n_mc in PASS_GRID:
    print(f'\n=================== N_MC = {n_mc} ===================')
    t_start = time.time()
    
    # 1. Center 1 (OOD val) - Compute U and tau (70th pct)
    src_m = load_fresh()
    u_val_list = []
    t_mc_val0 = time.time()
    for x, y, _ in tqdm(val_loader, desc=f'Val MC N={n_mc}'):
        x = x.to(device)
        _, var = mc_predict(src_m, x, n_mc)
        u_val_list.append(var.cpu().numpy())
    u_val = np.concatenate(u_val_list)
    tau = float(np.quantile(u_val, 0.70))
    print(f'Frozen tau (70th pct of Center-1 U) = {tau:.8g}')
    
    # 2. Center 2 (Target test) - MC prediction & TTA adaptation
    m = load_fresh()
    opt = torch.optim.Adam(bn_affine(m), lr=TTA_LR)
    ys, pb, pa, us = [], [], [], []
    
    t_mc0 = time.time()
    for x, y, _ in tqdm(test_loader, desc=f'Test TTA N={n_mc}'):
        x = x.to(device); B = x.size(0)
        with torch.no_grad():
            eval_mode(src_m)
            p0 = torch.sigmoid(src_m(x)).view(-1)
            pb.append(p0.cpu().numpy())
            _, var = mc_predict(src_m, x, n_mc)
            w = (var < tau).float()
            us.append(var.cpu().numpy())
        
        tta_mode(m)
        k = int(w.sum().item())
        if k > 0:
            opt.zero_grad(set_to_none=True)
            loss = (w * entropy(m(x))).sum() / w.sum()
            loss.backward()
            opt.step()
        
        with torch.no_grad():
            eval_mode(m)
            pa.append(torch.sigmoid(m(x)).view(-1).cpu().numpy())
        ys.append(y.view(-1).numpy())
        
    t_end = time.time()
    yv, b, a, uv = np.concatenate(ys), np.concatenate(pb), np.concatenate(pa), np.concatenate(us)
    u_test_dict[n_mc] = uv
    mask_test_dict[n_mc] = (uv < tau)
    
    met = metrics_of(yv, a)
    harm = flip_table(yv, b, a)
    cov = float(np.mean(uv < tau))
    
    res = {
        'MC_passes': n_mc,
        'tau': tau,
        'target_coverage': cov,
        'AUROC': met['auroc'],
        'F1': met['f1'],
        'ECE': met['ece'],
        'Harm_rate': harm['harm_rate'],
        'Correction_rate': harm['correction_rate'],
        'Runtime_s': t_end - t_start
    }
    results.append(res)
    print(f"N={n_mc:2d} | AUROC={met['auroc']:.4f} | F1={met['f1']:.4f} | ECE={met['ece']:.4f} | Harm={harm['harm_rate']*100:.2f}% | Time={t_end-t_start:.1f}s")

df_res = pd.DataFrame(results)
df_res.to_csv(WORK / 'mc_sensitivity_s42.csv', index=False)
print('\nSaved results to mc_sensitivity_s42.csv')

In [ ]:
# Uncertainty stability analysis & comparisons to N=20
u20 = u_test_dict[20]
m20 = mask_test_dict[20]
stability = []

for n_mc in PASS_GRID:
    un = u_test_dict[n_mc]
    mn = mask_test_dict[n_mc]
    rho, _ = stats.spearmanr(un, u20)
    jaccard = float(np.sum(mn & m20) / np.sum(mn | m20))
    stability.append({'MC_passes': n_mc, 'Spearman_vs_N20': rho, 'Overlap_vs_N20': jaccard})
    print(f'N={n_mc:2d} vs N=20: Spearman rho = {rho:.4f}, Overlap Jaccard = {jaccard:.4f}')